# nb00 — Extract the Managed Care Performance Monitoring Dataset (HEDIS + companions)

<hr style="border: 3px solid black;">

**Purpose:** pull every CSV resource of the DHCS Managed Care Performance Monitoring Dashboard Report from the CalHHS Open Data Portal (CKAN API), save raw copies, and profile each file so we can decide which ones feed blog post 2.

**Dataset:** https://data.chhs.ca.gov/dataset/managed-care-performance-monitoring-dashboard-report

**Known shape (from the portal, July 2026):** 9 CSV resources: HEDIS, Population, Age, Sex, Ethnicity, Provider Ratios, Encounter Completeness, Grievance Demographics, Grievance Type; plus 1 PDF report. Coverage 2017 to 2023, statewide, quarterly releases.

**Pipeline position:** nb00 (extract + profile) → nb01 (clean the chosen files, to be designed after this profile)

## 1. Setup

In [1]:
import json
from datetime import date
from pathlib import Path

import pandas as pd
import requests

CKAN_BASE = 'https://data.chhs.ca.gov/api/3/action'
DATASET_SLUG = 'managed-care-performance-monitoring-dashboard-report'

DATA_DIR = Path('..') / 'data'
RAW_DIR = DATA_DIR / 'raw'
RAW_DIR.mkdir(parents=True, exist_ok=True)
print('Raw folder:', RAW_DIR.resolve())

Raw folder: /Users/trinidadcisneros/Documents/Development/Coding/bitterscientist.com/bitterscientist.com/folders/ds_blogs/projects/tableau_hedis/data/raw


## 2. List every resource via the CKAN API

In [2]:
resp = requests.get(f'{CKAN_BASE}/package_show', params={'id': DATASET_SLUG}, timeout=60)
resp.raise_for_status()
pkg = resp.json()['result']

print('Dataset:', pkg['title'])
print('Resources:')
for r in pkg['resources']:
    print(f"  [{r.get('format','?'):4}] {r['name']:28} last_modified={r.get('last_modified')}")

Dataset: Managed Care Performance Monitoring Dashboard Report
Resources:
  [ZIP ] All resource data            last_modified=2025-11-07T02:55:58.867300
  [PDF ] Managed Care Performance Monitoring Dashboard Report last_modified=2024-04-25T17:18:40.916082
  [CSV ] HEDIS                        last_modified=2024-04-25T17:11:58.434597
  [CSV ] Population                   last_modified=2024-04-25T17:13:46.969773
  [CSV ] Provider Ratios              last_modified=2024-04-25T17:15:50.295236
  [CSV ] Sex                          last_modified=2024-04-25T17:17:35.735800
  [CSV ] Age                          last_modified=2024-04-25T17:20:55.190976
  [CSV ] Encounter Completeness       last_modified=2024-04-25T17:23:31.680066
  [CSV ] Ethnicity                    last_modified=2024-04-25T17:25:21.204857
  [CSV ] Grievance Demographics       last_modified=2024-04-25T17:28:45.516260
  [CSV ] Grievance Type               last_modified=2024-04-25T17:27:02.701388


## 3. Download every CSV resource

Raw files are immutable inputs. Filenames get the resource name (slugified) plus today's date.

In [3]:
import re

def slug(name):
    return re.sub(r'[^a-z0-9]+', '_', name.lower()).strip('_')

downloaded = []
for r in pkg['resources']:
    if r.get('format', '').upper() != 'CSV':
        continue
    out = RAW_DIR / f"{slug(r['name'])}_raw_{date.today().isoformat()}.csv"
    dl = requests.get(r['url'], timeout=300)
    dl.raise_for_status()
    out.write_bytes(dl.content)
    downloaded.append({'resource': r['name'], 'file': out.name,
                       'source_url': r['url'], 'last_modified': r.get('last_modified'),
                       'size_mb': round(out.stat().st_size / 1_048_576, 2)})
    print(f"saved {out.name} ({downloaded[-1]['size_mb']} MB)")

log = {'extracted_on': date.today().isoformat(), 'dataset': pkg['title'], 'files': downloaded}
(DATA_DIR / 'extraction_log.json').write_text(json.dumps(log, indent=2))
print('extraction_log.json written')

saved hedis_raw_2026-07-07.csv (0.01 MB)
saved population_raw_2026-07-07.csv (0.04 MB)
saved provider_ratios_raw_2026-07-07.csv (0.01 MB)
saved sex_raw_2026-07-07.csv (0.01 MB)
saved age_raw_2026-07-07.csv (0.03 MB)
saved encounter_completeness_raw_2026-07-07.csv (0.04 MB)
saved ethnicity_raw_2026-07-07.csv (0.04 MB)
saved grievance_demographics_raw_2026-07-07.csv (0.0 MB)
saved grievance_type_raw_2026-07-07.csv (0.01 MB)
extraction_log.json written


## 4. Profile every downloaded file

Shape, columns, and example values for each, so we can pick the blog's tables and plan nb01's cleaning.

In [4]:
for item in downloaded:
    path = RAW_DIR / item['file']
    try:
        df = pd.read_csv(path)
    except Exception as e:
        print(f"{item['resource']}: READ ERROR {e}")
        continue
    print('=' * 70)
    print(f"{item['resource']}  |  shape {df.shape}")
    for col in df.columns:
        examples = df[col].dropna().unique()[:3]
        print(f"   {col} ({df[col].nunique()} unique) e.g. {list(examples)}")

HEDIS  |  shape (436, 3)
   HEDIS Reporting Year (8 unique) e.g. [2016, 2017, 2018]
   Reporting Unit (56 unique) e.g. ['AAH - Alameda', 'Anthem - Alameda', 'Anthem - Contra Costa']
   AQFS (274 unique) e.g. ['53.18%', '44.55%', '45.00%']
Population  |  shape (648, 5)
   Month (12 unique) e.g. [202301, 202302, 202303]
   Measure Category (4 unique) e.g. ['Eligibility', 'New Enrollments', 'State Fair Hearings']
   Measure Type (9 unique) e.g. ['Member Months', 'New Enrollments', 'State Fair Hearings']
   Population Type (6 unique) e.g. ['Dual', 'Invalid', 'MO-ACA']
   Count (459 unique) e.g. ['1,345,887', '0', '36']
Provider Ratios  |  shape (372, 4)
   Plan Parent Reporting Name (31 unique) e.g. ['AAH', 'Aetna', 'AHF']
   Month (12 unique) e.g. [202302, 202303, 202304]
   PCPs per 2,000 Members (52 unique) e.g. [3, 22, 23]
   Physicians per 1,200 Members (124 unique) e.g. ['65', '64', '63']
Sex  |  shape (216, 5)
   Month (12 unique) e.g. [202301, 202302, 202303]
   Measure Category (4

## 5. Focused HEDIS profile

The HEDIS file is the core of post 2. Check: which measures, which years, which plans, how rates are stored, and whether benchmark columns (MPL/HPL) exist.

In [5]:
hedis_file = next(RAW_DIR / i['file'] for i in downloaded if 'hedis' in i['file'])
h = pd.read_csv(hedis_file)
print('Shape:', h.shape)
print()
print(h.dtypes)
h.head(10)

Shape: (436, 3)

HEDIS Reporting Year     int64
Reporting Unit          object
AQFS                    object
dtype: object


,HEDIS Reporting Year,Reporting Unit,AQFS
0,2016,AAH - Alameda,53.18%
1,2016,Anthem - Alameda,44.55%
2,2016,Anthem - Contra Costa,45.00%
3,2016,Anthem - Fresno,44.09%
4,2016,Anthem - Kings,46.82%
5,2016,Anthem - Madera,51.36%
6,2016,Anthem - Region 1,42.73%
7,2016,Anthem - Region 2,40.00%
8,2016,Anthem - Sacramento,45.91%
9,2016,Anthem - San Benito,38.18%


In [6]:
# Value inventories for the likely key columns (adjust names after seeing the profile above)
for col in h.columns:
    n = h[col].nunique()
    if n <= 40:
        print(f'--- {col} ({n}) ---')
        print(sorted(h[col].dropna().unique().astype(str))[:40])
        print()

--- HEDIS Reporting Year (8) ---
['2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023']



---

**Next:** paste the section 4 and 5 outputs into the working session; nb01 (cleaning) gets designed from what they show.